In [ ]:
import nba_api_module as nbpi 

#nbpi.get_season_game_log('2025-26')
#nbpi.get_game_snapshots('0022500045')

In [ ]:
# GameID metadata

from nba_api.stats.endpoints import boxscoretraditionalv3

game_id = "0022500045"

bs = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id)
d = bs.get_dict()

time = d['meta']['time']
print(time)

bst = d['boxScoreTraditional']

print(bst['homeTeam']['teamName'])
print(bst['awayTeam']['teamName'])

In [7]:
from nba_api.stats.endpoints import teamdashboardbygeneralsplits
import pandas as pd

team_id = 1610612747   # pl. Los Angeles Lakers
season = "2023-24"

# lekérés
dashboard = teamdashboardbygeneralsplits.TeamDashboardByGeneralSplits(
    team_id=team_id,
    season=season,
    season_type_all_star="Regular Season"
)

# a válasz struktúrája
data = dashboard.get_dict()

# Az NBA API több táblát ad vissza — a "OverallTeamDashboard" lesz a fő
overall = data["resultSets"][0]

headers = overall["headers"]
rows = overall["rowSet"]

df = pd.DataFrame(rows, columns=headers)

display(df.head())


,GROUP_SET,GROUP_VALUE,SEASON_YEAR,GP,W,L,W_PCT,MIN,FGM,FGA,...,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK
0,Overall,2023-24,2023-24,82,47,35,0.573,3971.0,3580,7177,...,1,1,1,1,1,1,1,1,1,1


In [15]:
from nba_api.stats.endpoints import leaguegamelog
import pandas as pd

def get_games(season="2023-24"):
    log = leaguegamelog.LeagueGameLog(
        season=season,
        season_type_all_star="Regular Season"
    )
    data = log.get_dict()
    
    df = pd.DataFrame(
        data['resultSets'][0]['rowSet'],
        columns=data['resultSets'][0]['headers']
    )
    
    # dátum konvertálása
    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
    return df

df = get_games("2023-24")

start = pd.to_datetime("2024-01-01")
end   = pd.to_datetime("2024-02-01")

filtered = df[
    (df["TEAM_ID"] == 1610612747) &
    (df["GAME_DATE"] >= start) &
    (df["GAME_DATE"] <= end)
]

team_stats = filtered.groupby("TEAM_ID").agg({
    "PTS": "mean",
    "FGM": "mean",
    "FGA": "mean",
    "REB": "mean",
    "AST": "mean",
    "PLUS_MINUS": "mean"
}).reset_index()

print(team_stats)

      TEAM_ID       PTS      FGM      FGA      REB    AST  PLUS_MINUS
0  1610612747  120.1875  44.1875  88.0625  41.8125  29.75     -2.6875
